**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Artificial Neural Networks

We build a neural network **from scratch in NumPy** — every forward pass, every gradient, every update written by hand — and train it until a spiral no line could separate falls to a few dozen lines of code. After this, [PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) will feel like a convenience, not a mystery.

## 0. Introduction

An artificial neuron computes $\sigma(\mathbf{w}^T\mathbf{x} + b)$: a weighted vote followed by a nonlinear squeeze. One neuron draws a single line through the data. The story of this workshop: *stack* votes into layers and the network bends that line into any boundary you need.

## 1. Pre-requisites

- [Intro to Python](../../Intro_Programming/Intro_Python/Intro_Python.ipynb) — NumPy fluency.
- The chain rule from calculus — backpropagation *is* the chain rule, organized.
- [Adaptive Filtering: APA](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) is a helpful cousin: LMS is literally training a single linear neuron.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)

---
### 🕐 Session 1 of 3 — *From Neuron to Network* (~35 min)
**Goal:** understand the perceptron, why nonlinearity is essential, and why depth buys expressiveness.
**Feeds into:** Session 2 (backpropagation).

---

## 2. Theory: Neurons, Layers, Nonlinearity

### 2.1. The Dataset That Defeats a Line

Two interleaved spirals. No single weighted vote — no *line* — can separate them.

In [ ]:

# YOUR CODE HERE


**What just happened.** Two interleaved arms, 300 points each, standardized to zero mean and unit variance. Take a moment and actually try to draw a separating line — any line. Whatever you choose, it cuts through both spirals, because each arm wraps around the origin through more than a full turn and revisits every direction.

**That failure is structural, not a matter of finding a better line.** A single neuron computes $\sigma(\mathbf{w}^Tx + b)$, and the set where it flips its vote is $\mathbf{w}^Tx + b = 0$ — a straight line, full stop. There are only three free numbers and none of them can produce a curve. So a perceptron on this data is capped at roughly 50% accuracy: chance.

**Note that the difficulty has nothing to do with size.** 600 points, two dimensions, no missing values, mild noise, perfectly balanced classes. By any modern measure this is a tiny, clean dataset. It is hard for exactly one reason — **the decision boundary is not linear** — and that single property is what motivates every layer added from here on.

**Two details in the generator are worth pointing at, since students will reuse this code.** The two classes are the *same* spiral with a $\pi$ phase offset, so they are perfectly interleaved by construction rather than by luck; and the final standardization is not cosmetic. He initialization (`sqrt(2/fan_in)` in the next cell) assumes inputs of roughly unit scale, and the raw spiral runs out to radius $3\pi \approx 9.4$. Feed the unstandardized data in and the first layer saturates immediately. **Standardization is a precondition for the initialization scheme, not a nicety.**

**Keep this picture in view for the rest of the workshop, because it is the yardstick.** In Session 3 the same plane gets painted by a trained network's predictions, and the boundary follows both arms around. Nothing changes about the data between here and there — what changes is that 32 hidden units have folded the space until a line in the final layer is enough.

💡 **Intuition.** Why nonlinearity is non-negotiable: stacking *linear* layers collapses — a matrix times a matrix is just another matrix, so a 100-layer linear network is one line in disguise. The activation function between layers breaks that collapse. With it, each hidden neuron contributes one *fold* of the input space; layers of folds crumple the plane until the spirals become linearly separable in the last layer.

### 2.2. Activations

We'll use **ReLU** ($\max(0, u)$) in hidden layers — cheap, and its gradient doesn't vanish for active units — and a **sigmoid** on the output to read the result as a probability.

In [ ]:

# YOUR CODE HERE


**What just happened.** Two very different shapes. ReLU is a hinge — flat zero to the left, slope exactly 1 to the right. The sigmoid is a smooth S, squashing all of $\mathbb{R}$ into $(0,1)$ and flattening out at both ends.

**Read them by their *derivatives*, because that is what backprop multiplies.** ReLU's derivative is 1 wherever the unit is active and 0 where it is not — so an active path passes the gradient through **unattenuated**. The sigmoid's derivative is $\sigma(1-\sigma)$, which peaks at **0.25** at the origin and falls toward zero at both extremes. That single number is the reason these two functions have different jobs.

**Multiply it out and the historical argument writes itself.** Stack ten sigmoid layers and the backward pass multiplies at most $0.25$ ten times: $0.25^{10} \approx 10^{-6}$. The early layers receive a gradient a million times smaller than the late ones and effectively stop learning. **That is the vanishing-gradient problem**, and it is why networks deeper than a few layers were impractical for decades. ReLU's slope of 1 removes the attenuation entirely, and deep learning became trainable.

**Which is why the two appear in different places, deliberately.** ReLU sits in the hidden layers, where the only requirement is a nonlinearity that does not strangle the gradient. Sigmoid sits on the *output*, where we genuinely want a number in $(0,1)$ to read as $P(\text{class }1)$ — and where saturation costs nothing, because there is no layer beneath it to starve.

**Note the price ReLU pays for that flat region.** A unit whose pre-activation is negative for **every** training example receives exactly zero gradient, forever, and can never recover — the "dying ReLU". It is a real failure mode, not a footnote; Leaky ReLU and GELU exist to give the negative side a small nonzero slope precisely to avoid it. Flatness is what makes the gradient clean on one side and fatal on the other.

**And one thing neither plot shows, which is the actual point of nonlinearity.** These are not chosen for their curves but for being *not linear*. Any of ReLU, sigmoid, tanh, GELU breaks the collapse $W_2(W_1x) = (W_2W_1)x$ and makes depth mean something. The differences between them are matters of gradient flow and cost — the requirement they all satisfy is the one from the previous cell, and it is the only requirement that is non-negotiable.

---
### 🕐 Session 2 of 3 — *Backpropagation* (~35 min)
**Goal:** derive the gradients as the chain rule on a computational graph — and implement them.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (training).

---

## 3. Theory: Backpropagation

💡 **Intuition.** Backprop is *blame assignment*. The loss says "prediction off by this much." Walking backward through the network, each operation answers one local question: *given how much my output was to blame, how much were my inputs and weights to blame?* — that answer is its local derivative. Multiplying local blames along the path is exactly the chain rule; "backprop" is just doing it once per node, back-to-front, instead of re-deriving a formula per weight.

### 3.1. The Two-Layer Network

Forward pass, batch $X$ ($n \times 2$):

$$Z_1 = X W_1 + \mathbf{b}_1 \quad A_1 = \mathrm{ReLU}(Z_1) \quad Z_2 = A_1 W_2 + \mathbf{b}_2 \quad \hat{y} = \sigma(Z_2)$$

Loss — binary cross-entropy: $\;L = -\frac{1}{n}\sum y\log\hat{y} + (1-y)\log(1-\hat{y})$

### 3.2. The Gradients

The famous simplification: for sigmoid + cross-entropy, the output blame collapses to $\delta_2 = \hat{y} - y$ (predicted minus true — the *error*, again!). Then

$$\nabla W_2 = \tfrac{1}{n} A_1^T \delta_2 \qquad \delta_1 = (\delta_2 W_2^T) \odot \mathbf{1}[Z_1 > 0] \qquad \nabla W_1 = \tfrac{1}{n} X^T \delta_1$$

The ReLU mask $\mathbf{1}[Z_1>0]$ says: neurons that were off take no blame.

In [ ]:

# YOUR CODE HERE


### 3.3. Trust, but Verify: the Gradient Check

The classic backprop bug is a silently wrong gradient. The antidote: compare against a finite difference $\frac{L(\theta + \epsilon) - L(\theta - \epsilon)}{2\epsilon}$ on a few random weights.

In [ ]:

# YOUR CODE HERE


**What just happened.** Three weights sampled from three different tensors, and analytic and numeric gradients agree to all six printed digits:

| parameter | analytic | numeric |
|---|---|---|
| `W1[0,3]` | −0.003470 | −0.003470 |
| `W2[7,0]` | −0.055097 | −0.055097 |
| `b1[2]` | +0.069507 | +0.069507 |

And the `assert` demands agreement below $10^{-6}$, so this is a **test that can fail**, not a printout to admire. The hand-derived backward pass is correct.

**Note that the three checks are not redundant — each exercises a different part of the derivation.** `W2` tests only the output layer and the $\delta_2 = \hat y - y$ collapse. `W1` tests the full path, including the ReLU mask and the $\delta_2W_2^T$ propagation — the step where sign errors and missing transposes actually live. `b1` tests the bias reduction, which is the one place a `sum` and a `mean` are easy to confuse. A check that only sampled `W2` would pass with a badly broken `W1`.

**Why this matters more than it appears: a wrong gradient does not crash.** It produces a network that trains a little slowly, plateaus at a mediocre loss, or converges somewhere odd — symptoms indistinguishable from "the model needs more capacity" or "the learning rate is off". Practitioners have lost days to a transposed matrix that ran perfectly. **The failure mode of backprop is silence**, and that is precisely why an explicit check earns its five lines.

**The $\varepsilon = 10^{-5}$ choice is a real numerical decision, not a magic number.** The central difference has truncation error $O(\varepsilon^2)$ and cancellation error $O(\varepsilon_{\text{mach}}/\varepsilon)$; the two balance near $\sqrt[3]{\varepsilon_{\text{mach}}} \approx 10^{-5}$ in double precision. Try $10^{-12}$ and the numerator becomes the difference of two nearly identical doubles — the check starts reporting large errors for a *correct* gradient, and the natural but wrong conclusion is that the gradient is broken.

**One caveat on ReLU that is worth knowing before students hit it.** ReLU is not differentiable at zero, so if a perturbation pushes some $Z_1$ entry across the hinge, the finite difference and the analytic gradient will genuinely disagree — not a bug in either, but a real kink in the function. With 600 examples and random weights it is unlikely enough not to matter here, and it is why gradient checks on ReLU networks are sometimes run with a smooth activation substituted in.

**Finally, note that this is the same discipline the frameworks institutionalise.** `torch.autograd.gradcheck` does exactly this against autograd's output, and it exists because even the people who wrote the autodiff engine do not trust hand-written backward methods without testing them. Writing the check yourself once is what makes that habit portable.

---
### 🕐 Session 3 of 3 — *Training the Network* (~40 min)
**Goal:** run the full training loop on the spiral, visualize the learned boundary, then meet PyTorch.
**Builds on:** Session 2. &nbsp; **Feeds into:** [Intro to PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb).

---

## 4. Application: Train on the Spiral

In [ ]:

# YOUR CODE HERE


**What just happened.** **100.0% training accuracy** on the dataset that defeated every line, and the loss curve shows how: a slow start, then a steep collapse, then a long grind toward zero. Thirty-two hidden units and 3000 plain gradient steps — no momentum, no Adam, no mini-batching, no regularisation, no framework.

**Read the curve's shape rather than its endpoint, because the shape is the lesson.** The early plateau is not the code failing to work; it is gradient descent on a non-convex surface, where the network has not yet found a representation that makes the spiral tractable and the gradient is genuinely small. Then the drop, when useful folds appear. Then diminishing returns, as the remaining loss comes from points near the boundary that the network is merely growing more confident about. **Students who abandon a run during the plateau abandon runs that would have worked.**

**Now the honest caveat, because 100% is exactly the number that should make you suspicious.** This is **training** accuracy, on the same 600 points the network optimised against, with **no held-out set anywhere in this notebook**. It measures memorisation capacity, not generalisation. A network with 32 hidden units has $2 \times 32 + 32 + 32 + 1 = 129$ parameters against 600 examples — comfortably enough to fit, and comfortably enough to fit the noise too. The correct claim is "the network can represent and fit this boundary", not "the network learned the spiral."

**Test the distinction rather than arguing it.** Generate a second spiral dataset with the same code and a fresh seed, then evaluate. Accuracy will drop — modestly here, because the boundary genuinely is a spiral and the model found it, but it will drop. That gap between train and test is the only quantity that ever tells you whether a model learned anything, and every workshop downstream of this one takes it as the primary metric.

**Two implementation notes worth catching while the numbers are on screen.** The loss is recorded every 50 epochs, so the curve has 60 points, not 3000 — the visible smoothness is partly sampling. And this is **full-batch** gradient descent: every step uses all 600 points, which is why a learning rate as large as 0.5 is stable. Mini-batch SGD injects gradient noise and would need a smaller rate or a schedule, which is the regime every real training script operates in.

**Finally, note what was *not* needed.** No adaptive optimiser, no batch norm, no dropout, no learning-rate schedule, no early stopping. The four-beat loop — predict, grade, diagnose, nudge — with a fixed step size was sufficient. Everything else in the modern toolkit exists to make this loop work on problems where it otherwise would not; none of it is a different idea.

In [ ]:
# Visualize what the network learned: paint every point of the plane by its prediction

# YOUR CODE HERE


**What just happened.** The whole plane is painted by the network's output, and the red/blue frontier **spirals** — following both arms around, threading between them, staying confident (deep colour) where data is dense and hedging toward 0.5 (pale) in the gaps. Compare it to the first plot in Session 1, where the challenge was to draw a separating line. This is that line, after 32 hidden units got hold of the plane.

**The key question to ask here: how did a stack of straight-line units produce a curve?** Every neuron computes $\mathbf{w}^Tx + b$ — nothing in the network can bend anything. The answer is Session 1's folding picture, made concrete. Each ReLU hinge splits the plane along a line, silent on one side; 32 hinges partition it into many polygonal regions, and within each region the network is exactly linear. The final layer draws **one straight line** in that folded coordinate system, and unfolding turns it into the spiral you see.

**Which means the boundary is piecewise linear, not smooth — and that is checkable.** Zoom in far enough and the spiral resolves into flat facets meeting at corners. A ReLU network can only ever produce a piecewise-linear function; its apparent smoothness is a matter of having more pieces than pixels. Worth stating plainly, because "neural networks learn smooth functions" is a common and false belief, and the piecewise structure explains real behaviour (adversarial examples live on those facets).

**Read the pale regions as the model's uncertainty, with a caveat.** Between the arms, where no training point falls, the output drifts toward 0.5 — the network hedges where it has no evidence. That is the desirable behaviour. But look at the corners of the plot, far outside the data: the network is *confident* out there, deeply coloured, on the basis of nothing at all. **Extrapolation confidence is not calibrated uncertainty.** A ReLU network extends its outermost linear pieces to infinity, so it will always have an opinion about regions it has never seen, and that opinion is an artifact of the folds rather than a claim about the world. This is exactly why [Uncertainty in ML](../Uncertainty_in_ML.ipynb) exists as its own workshop.

**Note also what the boundary reveals about capacity.** It hugs the arms without growing fingers toward individual points — 32 units is roughly the right size for this problem. Re-run with width 4 and the boundary is too coarse to follow both turns; with 256 it starts reaching for isolated noisy points, and the frontier acquires blobs and spurs that correspond to nothing real. Same bias–variance curve as the projection dimension in [Linear Algebra](../../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) Session 2, now measured in parameters.

**And the cheapest experiment in the workshop lives right here.** Set `A1 = Z1` in `forward`, retrain, and re-plot. The boundary collapses to a single straight line and accuracy falls to roughly 50%, because $W_2(W_1x)$ is just another matrix. Thirty seconds of work that turns Session 1's central claim from an assertion into a result you watched happen.

Experiments worth 5 minutes each (edit and re-run):

- Hidden width 4 vs 32 vs 256 — watch the boundary sharpen (and eventually overfit the noise).
- Remove the ReLU (`A1 = Z1`) — the boundary collapses to a line, *proving* Session 1's claim.
- Learning rate 5.0 — meet divergence in person.

## 5. Conclusion

Everything deep learning does was in this notebook: forward pass, loss, chain-rule blame assignment, gradient step. Frameworks add autograd (no hand-derived gradients), GPU tensors, and libraries of layers — conveniences on top of *this* loop.

---
## Where next

- [Intro to PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) — the same network with autograd doing Session 2 for you.
- [Convolutional Neural Networks](../README.md#workshop-2--convolutional-neural-networks-available) — weight sharing turns layers into learned filter banks (bridging back to [DSP](../../Intro_DSP/README.md)).
- [Intro to Transformers](../../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — attention as data-dependent connectivity.